# 08 | Marketing allocation and channel feasibility

**Author: Chanakya**

Meet the brief with reconciled percentages, explicit cost builds and conditional CACs. The case supplies target CAC and current spending shares. Proposed envelopes and incremental-response scenarios are separate from those supplied facts.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/manifests/release.json').exists())
sys.path.insert(0, str(ROOT))
from src.analysis_common import *
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
rng = np.random.default_rng(CFG['seed'])
print('Offline inputs:', CFG['raw_release'], '| Author: Chanakya')
import src.analysis_common as shared
shared.ACTIVE_NOTEBOOK='08_channel_economics'
shared.ACTIVE_SOURCES=['kofluence_2025_report', 'v3_meta_inr_ratecard_jan2026', 'v3_nurdd_india']

## 0. Supplied baseline and CAC definitions
The brief gives the INR150–200 target per paying subscriber, current performance spend of 65–70%, a recurring 7.8% contest click-to-purchase rate, and an engagement peak around D-2. ESPN.in takeovers and native personalities are existing levers. Native personalities have effectively zero incremental media-rights cost. Production and distribution costs remain separate. The absolute marketing budget and achieved channel CAC are not supplied. The case does not define its target as causal incremental CAC. Purchase CAC and incremental CAC must therefore be reported separately. Applying INR150–200 to incremental CAC is an additional proposed hurdle, not a reinterpretation of the supplied target. INR175 is a scenario within the target range, not reported performance.

In [ ]:
baseline=pd.DataFrame([('Performance marketing',.65,.70),('Brand and other acquisition',.30,.35)],columns=['case_bucket','current_share_low','current_share_high']);display(table(baseline,'08_current_spend_baseline'))
cpc_gate=pd.DataFrame([dict(target_cac=t,contest_click_to_purchase=.078,maximum_cpc_before_other_cost=t*.078) for t in [150,200]]);display(table(cpc_gate,'08_case_contest_cpc_ceiling'))
print('At supplied 7.8% conversion, media-only purchase CAC = CPC / 0.078. Target INR150–200 permits INR11.70–15.60 CPC before partner, prize, creative and other acquisition costs.')
print('The proposed 40/20/20/10/10 groups are all-in operating envelopes. They are not directly comparable with the current performance-versus-brand accounting split. Reclassify all paid distribution consistently before claiming a change in performance share.')

## 1. Normalize the budget and expose each cost component
Use INR100,000 as a planning scale. Fixed production, staff and prize budgets are management assumptions. CPC and messaging rates are external proxies. The stress model counts incremental new FanCode payers. A separate table reports purchase CAC where the scenario provides that denominator. Cross-channel overlap must be prevented by disjoint cells or measured with a combined holdout.

In [ ]:
allocation=pd.DataFrame([dict(channel=k,share=v,budget=CFG['pilot_budget']*v) for k,v in CFG['allocation'].items()]);display(table(allocation,'08_budget_allocation'))
components=pd.DataFrame([
('Paid occasions','Creative and operations',6000,'Planning assumption',''),('Paid occasions','Media envelope',34000,'Budget remainder','v3_nurdd_india'),
('Owned lifecycle','Creative, tooling and operations',12000,'Planning assumption',''),('Owned lifecycle','Messaging envelope',8000,'Budget remainder','v3_meta_inr_ratecard_jan2026'),
('Communities and contests','Partner, prizes and operations',12000,'Planning assumption, no partner quote',''),('Communities and contests','Click acquisition envelope',8000,'Budget remainder','v3_nurdd_india'),
('Native creators','Four nano reels at INR1500',6000,'Chosen fee within broad external range','kofluence_2025_report'),('Native creators','Usage, editing and distribution',4000,'Planning assumption',''),('Measurement','Experiment setup and analysis',10000,'Planning assumption','')],columns=['channel','component','cost','status','source_id'])
display(table(components,'08_cost_build'))
print('WhatsApp external rate:',METRICS['whatsapp_marketing']['value'],'INR/message. BSP markup, current contract and tax treatment are not verified.')
print('Paid CPC proxy:',METRICS['meta_cpc_d2c']['value'],'INR/click, unrelated D2C categories. No FanCode audience quote.')

## 2. Conditional outcomes and costs for each acquisition method
Use three stress-test cells, not probability-weighted forecasts. Paid conversion is assumed, contest conversion is the supplied 7.8% with uncertain transferability, owned response is direct incremental percentage-point lift, and creator attributed clicks are assumed capacity. The optimistic cell combines favourable inputs and is not a confidence limit.

In [ ]:
scenarios=[dict(scenario='Adverse',cpc=18,paid_cvr=.005,incremental_share=.3,owned_lift=.002,creator_clicks=100,creator_cvr=.01),dict(scenario='Working test',cpc=10,paid_cvr=.015,incremental_share=.6,owned_lift=.01,creator_clicks=500,creator_cvr=.02),dict(scenario='Favourable',cpc=5,paid_cvr=.03,incremental_share=.9,owned_lift=.03,creator_clicks=1500,creator_cvr=.05)]
rows=[];rate=float(METRICS['whatsapp_marketing']['value'])
for s in scenarios:
 outcome={
 'Paid occasions':(34000/s['cpc'],34000/s['cpc']*s['paid_cvr']*s['incremental_share'],'paid clicks'),
 'Owned lifecycle':(8000/rate,8000/rate*s['owned_lift'],'one delivered message per eligible prospect'),
 'Communities and contests':(8000/s['cpc'],8000/s['cpc']*.078*s['incremental_share'],'contest clicks'),
 'Native creators':(s['creator_clicks'],s['creator_clicks']*s['creator_cvr']*s['incremental_share'],'creator attributed clicks')}
 for channel,(quantity,payers,unit) in outcome.items():
  cost=float(allocation.set_index('channel').loc[channel,'budget']);rows.append(dict(scenario=s['scenario'],channel=channel,quantity=quantity,unit=unit,cost=cost,incremental_payers=payers,icac=cost/payers,required_channel_ceiling=180,meets_180=cost/payers<=180))
channels=pd.DataFrame(rows)
comparisons=[]
for row in channels.itertuples():
 q=next(x['incremental_share'] for x in scenarios if x['scenario']==row.scenario)
 gross=row.incremental_payers/q if row.channel!='Owned lifecycle' else np.nan
 comparisons.append(dict(scenario=row.scenario,channel=row.channel,cost=row.cost,attributed_pass_purchases=gross,purchase_cac=row.cost/gross if pd.notna(gross) else np.nan,incremental_payers=row.incremental_payers,incremental_cac=row.icac,denominator_note='Purchases must be deduplicated and confirmed as acquired paying subscribers for direct target comparison' if pd.notna(gross) else 'Owned scenario specifies uplift only, total purchase conversion is not supplied'))
display(table(pd.DataFrame(comparisons),'08_purchase_and_incremental_cac'))
table(pd.DataFrame(scenarios),'08_response_assumptions');display(table(channels,'08_conditional_channel_cac'));table(channels,'channel_scenarios',True)
blend=channels.groupby('scenario').agg(acquisition_cost=('cost','sum'),incremental_payers=('incremental_payers','sum')).reset_index();blend['measurement_cost']=10000;blend['fully_loaded_icac']=(blend.acquisition_cost+blend.measurement_cost)/blend.incremental_payers;display(table(blend,'08_blended_programme_cac'))
plt.figure(figsize=(10,4));pivot=channels.pivot(index='channel',columns='scenario',values='icac');pivot.plot.barh(ax=plt.gca(),logx=True);plt.axvline(180,color='black',ls='--');plt.xlabel('Conditional incremental CAC (INR, logarithmic axis)');plt.title('The initial allocation is a learning budget, not proven efficiency');plt.legend(fontsize=8);fig('08_channel_cac','External unit-cost proxies plus explicit response assumptions. INR180 channel gate allows a 10% measurement reserve.')

## 3. Solve for feasible procurement and response
A rate is useful only with a denominator. Under the stated budget, owned messaging also has a reach ceiling. Publisher takeovers lack a quote, so show allowable fixed fees under explicit click scenarios. The takeover is an alternative use of the community allocation, never an extra unbudgeted channel.

In [ ]:
gates=[]
for target in [150,200]:
 cell=target*.9
 for q in [.3,.6,.9,1]:
  for clicks in [500,1000,2000]:
   gross=clicks*.078*q;gates.append(dict(programme_target=target,channel_ceiling=cell,incremental_share=q,clicks=clicks,maximum_all_in_contest_or_takeover_fee=gross*cell,maximum_media_cpc_before_fixed=cell*.078*q,required_clicks_for_20000=20000/(cell*.078*q)))
display(table(pd.DataFrame(gates),'08_contest_takeover_procurement_gates'))
creators=pd.DataFrame([dict(reel_fee=fee,required_incremental_payers_fee_only=np.ceil(fee/180),source_id='kofluence_2025_report',limitation='Broad nano-creator range, excludes usage/editing/amplification') for fee in [500,1500,5000]]);display(table(creators,'08_creator_fee_hurdles'))
owned=pd.DataFrame([dict(target=target,incremental_payers_required=20000/(target*.9),messages_at_quoted_rate=8000/rate,minimum_incremental_conversion_lift=(20000/(target*.9))/(8000/rate)) for target in [150,200]]);display(table(owned,'08_owned_capacity_gate'))
reserve=pd.DataFrame([dict(reserve_share=r,target=t,acquisition_cell_ceiling=t*(1-r),programme_payers_required=np.ceil(100000/t)) for r in [0,.05,.1,.2] for t in [150,200]]);table(reserve,'08_measurement_reserve_sensitivity')
check('08_channels',{'allocation_totals_100_percent':abs(allocation.share.sum()-1)<1e-12,'budget_reconciles':components.cost.sum()==100000,'component_channel_totals_match':components.groupby('channel').cost.sum().sort_index().equals(allocation.set_index('channel').budget.sort_index().astype(int)),'harmonic_blend_not_mean':all(abs(r.fully_loaded_icac-100000/r.incremental_payers)<1e-8 for r in blend.itertuples()),'reserve_gate_180':200*.9==180,'contest_cpc_gate':abs(180*.078*.6-8.424)<1e-9})
report('08_channel_findings','Maintain the 40/20/20/10/10 split as a staged learning allocation, with stop/reallocation gates. Print numerical conditional CACs and their component assumptions together. Do not claim the working response cell is achievable. Owned reach is finite and non-zero-cost. A publisher takeover requires a quote below the conditional fee ceiling and attributable traffic verification, then a holdout for incremental payers. The normalized budget is not necessarily sufficient to power all experiments simultaneously.')

## Source references
These IDs resolve to the preserved bodies, URLs and capture timestamps. Derived tables also retain row-level source IDs where applicable. Case inputs refer to the supplied brief, physical PDF pages 9–14. Review source files resolve through the review collection log. Scenario parameters are in analysis_config.

In [ ]:
references=source_table(['kofluence_2025_report', 'v3_meta_inr_ratecard_jan2026', 'v3_nurdd_india'])
display(table(references,'08_source_references'))